In [ ]:
RUN_TARGET = "local"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1.
2. Run the install cell and restart if requested.
3. Mount Drive if you want to read synced `models/` and `results/` artifacts.
4. Run the remaining cells top to bottom.

### Running locally
1. Keep `RUN_TARGET = "local"`.
2. The notebook scans the local `models/` and `results/` folders.
3. It overlays all available probe artifacts on shared comparison plots.


In [ ]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "statsmodels": "0.14.5",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


# 07 - Cross-Model Probe Visualisation

This notebook scans `models/` for available XAllergen checkpoints, matches them to saved probe CSVs in `results/`, and overlays all available families on the same violin and density plots for comparison.

In [ ]:
import sys
from pathlib import Path
import importlib

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))

import baseline_notebook_utils
import mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from baseline_notebook_utils import configure_matplotlib_cache, detect_device, find_project_root, print_runtime_context
from mtl_epitope_notebook_utils import plot_probe_density_trends, plot_probe_violins

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd


In [ ]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Discover Available Variants

Each MTL checkpoint is mapped to its expected probe-row CSV. Missing probe outputs are skipped so the notebook can be rerun safely as more variants finish training.

In [ ]:
def infer_variant_from_checkpoint_name(checkpoint_name: str) -> tuple[str | None, str | None]:
    if checkpoint_name == "baseline_frozen_esm2.pt":
        return "baseline", "Baseline (04)"
    if checkpoint_name == "mtl_frozen_esm2_epitope.pt":
        return "frozen", "MTL (05 frozen)"
    if checkpoint_name.startswith("mtl_") and checkpoint_name.endswith("_esm2_epitope.pt"):
        variant = checkpoint_name[len("mtl_") : -len("_esm2_epitope.pt")]
        if variant:
            return variant, f"MTL ({variant})"
    return None, None


def probe_rows_path_for_variant(variant: str) -> Path:
    if variant == "baseline":
        return RESULTS_DIR / "baseline_probing_rows.csv"
    if variant == "frozen":
        return RESULTS_DIR / "mtl_probing_rows.csv"
    return RESULTS_DIR / f"mtl_{variant}_probing_rows.csv"


records = []
for checkpoint_path in sorted(MODELS_DIR.glob("*.pt")):
    variant, default_label = infer_variant_from_checkpoint_name(checkpoint_path.name)
    if variant is None:
        continue
    probe_rows_path = probe_rows_path_for_variant(variant)
    records.append(
        {
            "checkpoint_name": checkpoint_path.name,
            "variant": variant,
            "model_family": default_label,
            "checkpoint_path": checkpoint_path,
            "probe_rows_path": probe_rows_path,
            "probe_rows_exists": probe_rows_path.exists(),
        }
    )

discovery_df = pd.DataFrame(records)
discovery_df


In [ ]:
available_df = discovery_df[discovery_df["probe_rows_exists"]].copy()
if available_df.empty:
    raise FileNotFoundError(
        f"No matching probe-row CSVs were found in {RESULTS_DIR}. Run notebook 04, 05, or 06 first."
    )

family_label_overrides = {
    "baseline": "Baseline (04)",
    "frozen": "MTL (05 frozen)",
    "top1_unfrozen": "MTL (06 top1_unfrozen)",
}

frames = []
for row in available_df.itertuples(index=False):
    frame = pd.read_csv(row.probe_rows_path)
    family_label = family_label_overrides.get(row.variant, row.model_family)
    frame = frame.copy()
    frame["model_family"] = family_label
    frame["source_probe_rows_path"] = str(row.probe_rows_path)
    frames.append(frame)

all_probe_df = pd.concat(frames, ignore_index=True)
all_probe_df.head()


## Save Combined Tables

These artifacts provide a single row-wise table plus a compact mean and standard-deviation summary across all available families.

In [ ]:
MULTI_MODEL_ROWS_CSV = RESULTS_DIR / "all_models_probing_rows.csv"
MULTI_MODEL_SUMMARY_CSV = RESULTS_DIR / "all_models_probing_summary.csv"
MULTI_MODEL_VIOLINS_PNG = RESULTS_DIR / "all_models_probing_violins.png"
MULTI_MODEL_AUROC_PNG = RESULTS_DIR / "all_models_probing_auroc_vs_density.png"
MULTI_MODEL_AUPRC_PNG = RESULTS_DIR / "all_models_probing_auprc_vs_density.png"

all_probe_df.to_csv(MULTI_MODEL_ROWS_CSV, index=False)
summary_df = (
    all_probe_df.groupby(["model_family", "method"], as_index=False)
    .agg(
        auroc_mean=("auroc", "mean"),
        auroc_sd=("auroc", "std"),
        auprc_mean=("auprc", "mean"),
        auprc_sd=("auprc", "std"),
        precision_at_k_mean=("precision_at_k", "mean"),
        precision_at_k_sd=("precision_at_k", "std"),
        n_proteins=("accession", "count"),
    )
)
summary_df = summary_df.round({
    "auroc_mean": 4,
    "auroc_sd": 4,
    "auprc_mean": 4,
    "auprc_sd": 4,
    "precision_at_k_mean": 4,
    "precision_at_k_sd": 4,
})
summary_df.to_csv(MULTI_MODEL_SUMMARY_CSV, index=False)
summary_df


## Plot All Available Families

In [ ]:
plot_probe_violins(all_probe_df, MULTI_MODEL_VIOLINS_PNG)
plot_probe_density_trends(
    all_probe_df,
    MULTI_MODEL_AUROC_PNG,
    MULTI_MODEL_AUPRC_PNG,
)


In [ ]:
print("Saved combined artifacts:")
for out_path in [
    MULTI_MODEL_ROWS_CSV,
    MULTI_MODEL_SUMMARY_CSV,
    MULTI_MODEL_VIOLINS_PNG,
    MULTI_MODEL_AUROC_PNG,
    MULTI_MODEL_AUPRC_PNG,
]:
    print(f"  {out_path}")
